# Batch experiments
Set `test_mode = True` for one short run per configuration under `jobs/test/`, or `False` for the full experiment and Slurm scripts under `jobs/`. Either mode runs in an empty directory with only Python. Run the commands from the release directory containing the code and datasets.

## Experiment settings
Dataset IDs follow the order below. The number next to each dataset is its class count. Resistance is in ohms and capacitance in farads.

In [ ]:
test_mode = False
test_epochs = 10

settings = {
  "datasets": {"FreezerSmallTrain": 2, "MoteStrain": 2, "TwoLeadECG": 2, "PowerCons": 2, "CBF": 3, "ItalyPowerDemand": 2, "BME": 3, "Slope": 2},
  "seeds": range(30),
  "training_args": [
    "--task temporal --DataPath dataset --temporal-protocol learnable_filters",
    f"--temporal-encoding stored --voltage-range -1 1 --DEVICE cpu --EPOCH {test_epochs if test_mode else 10000000000}",
    "--PROGRESSIVE true --PATIENCE 500 --LR_PATIENCE 100 --LR_DECAY 0.5 --LR_MIN 0.0001",
    "--TIMELIMITATION 100000 --learnable_group w --FittingLoss ce --temporal-loss mean",
    "--temporal-readout sum --metric acc --N_train 1 --N_test 1 --e_train 0 --e_test 0",
    "--MODE nominal --powerestimator none"
  ],
  "common_circuit": {"version": 1, "dt": 0.002, "substeps": 1, "initial_voltage": 0.2, "valid_seed": 1001, "test_seed": 2001, "assume_ideal_drive": True},
  "templates": {
    "prnn-1": {
      "args": "--temporal-model prnn --activation-circuit tanh --circuit-sharing network --initial-state uniform --LR 0.1 --coupling-factor 0.1",
      "hidden": {"layers": 2, "scale": 1, "power": 1},
      "circuit_option": "--recurrent-config",
      "circuit": {}
    },
    "prnn-2": {
      "args": "--temporal-model prnn2 --activation-circuit tanh --circuit-sharing network --initial-state uniform --LR 0.1 --coupling-factor 0.1",
      "hidden": {"layers": 2, "scale": 1, "power": 1},
      "circuit_option": "--recurrent-config",
      "circuit": {}
    },
    "gated": {
      "args": "--temporal-model pltc --activation-circuit sigmoid --circuit-sharing neuron --initial-state uniform --LR 0.1 --initial-range 1.0 --gate-lr-scale 3.0 --target-lr-scale 2.0",
      "hidden": {"layers": 1, "scale": 2, "power": 2},
      "circuit_option": "--gated-config",
      "circuit": {"substeps": 3, "readout": "sigmoid", "target": "sigmoid", "gate": "sigmoid", "output": "sigmoid", "resistance": 1000000.0, "capacitance": 1e-06, "rc_variation": None, "assume_reciprocal": True, "assume_ideal_readout": True, "gate_input_share": 0.05, "gate_state_share": 0.1}
    },
    "rc-only": {
      "args": "--temporal-model pltc --activation-circuit sigmoid --circuit-sharing neuron --initial-state uniform --LR 0.1 --initial-range 1.0",
      "hidden": {"layers": 1, "scale": 2, "power": 2},
      "circuit_option": "--gated-config",
      "circuit": {"substeps": 3, "readout": "sigmoid", "target": "sigmoid", "gate": "sigmoid", "output": "sigmoid", "resistance": 1000000.0, "capacitance": 1e-07, "rc_variation": None, "assume_reciprocal": True, "assume_ideal_readout": True, "transistor_enabled": False}
    }
  },
  "projects": {
    "prnn-1": {"template": "prnn-1", "id": "prnn_ref"},
    "prnn-2": {"template": "prnn-2", "id": "prnn2_ref"},
    "pltc-ref": {"template": "gated", "id": "pltc_ref"},
    "pltc-grid-R0.1MOhm-C100nF": {"template": "gated", "id": "gateinit_c1e-07_r100000", "circuit": {"resistance": 100000.0, "capacitance": 1e-07}},
    "pltc-grid-R1MOhm-C100nF": {"template": "gated", "id": "gateinit_c1e-07_r1e+06", "circuit": {"resistance": 1000000.0, "capacitance": 1e-07}},
    "pltc-grid-R10MOhm-C100nF": {"template": "gated", "id": "gateinit_c1e-07_r1e+07", "circuit": {"resistance": 10000000.0, "capacitance": 1e-07}},
    "pltc-grid-R0.1MOhm-C10nF": {"template": "gated", "id": "gateinit_c1e-08_r100000", "circuit": {"resistance": 100000.0, "capacitance": 1e-08}},
    "pltc-grid-R1MOhm-C10nF": {"template": "gated", "id": "gateinit_c1e-08_r1e+06", "circuit": {"resistance": 1000000.0, "capacitance": 1e-08}},
    "pltc-grid-R10MOhm-C10nF": {"template": "gated", "id": "gateinit_c1e-08_r1e+07", "circuit": {"resistance": 10000000.0, "capacitance": 1e-08}},
    "pltc-grid-R0.1MOhm-C1nF": {"template": "gated", "id": "gateinit_c1e-09_r100000", "circuit": {"resistance": 100000.0, "capacitance": 1e-09}},
    "pltc-grid-R1MOhm-C1nF": {"template": "gated", "id": "gateinit_c1e-09_r1e+06", "circuit": {"resistance": 1000000.0, "capacitance": 1e-09}},
    "pltc-grid-R10MOhm-C1nF": {"template": "gated", "id": "gateinit_c1e-09_r1e+07", "circuit": {"resistance": 10000000.0, "capacitance": 1e-09}},
    "pltc-learned-r": {"template": "gated", "id": "pltc_learnable_tau", "circuit": {"learnable_resistance": True, "resistance_bounds": [1000.0, 1000000000.0]}},
    "pltc-constant-gate-input": {"template": "gated", "id": "pltc_constgate_learnable_tau", "circuit": {"constant_gate_input": 0.2, "learnable_resistance": True, "resistance_bounds": [1000.0, 1000000000.0]}},
    "pltc-rc-only-fixed-r": {"template": "rc-only", "id": "pltc_rconly_fixed"},
    "pltc-rc-only-learned-r": {"template": "rc-only", "id": "pltc_rconly_learnable_tau", "circuit": {"learnable_resistance": True, "resistance_bounds": [1000.0, 1000000000.0]}}
  }
}

In [ ]:
import json
import shlex
from pathlib import Path

commands_per_job = 16
if not isinstance(commands_per_job, int) or commands_per_job < 1:
    raise ValueError("commands_per_job must be a positive integer")
jobs = Path("jobs/test" if test_mode else "jobs")
jobs.mkdir(parents=True, exist_ok=True)
python = "python" if test_mode else "python3"
datasets = list(settings["datasets"].items())
seeds = settings["seeds"]
if test_mode:
    datasets = datasets[:1]
    seeds = [0]

train, evaluate = [], []
common = " ".join(settings["training_args"]).split()
for projectname, project in settings["projects"].items():
    if test_mode:
        projectname = f"{projectname}"
    template = settings["templates"][project["template"]]
    shape = template["hidden"]
    circuit = {**settings["common_circuit"], **template["circuit"], **project.get("circuit", {})}
    model_args = [*template["args"].split(), template["circuit_option"]]
    # CMD and Bash both pass this double-quoted, escaped JSON as one argument.
    circuit_arg = json.dumps(json.dumps(circuit, separators=(",", ":")))
    for dataset_id, (name, classes) in enumerate(datasets):
        width = shape["scale"] * classes ** shape["power"]
        network_args = ["--dataset-name", f"Dataset_{name.lower()}.tsds",
                        "--hidden", *([str(width)] * shape["layers"])]
        for seed in seeds:
            run = ["--DATASET", str(dataset_id), "--SEED", str(seed), "--projectname", projectname]
            command = shlex.join([python, "experiment.py", *run, *common, *model_args])
            remaining = shlex.join(["--sweep-config", project["id"], *network_args])
            train.append(f"{command} {circuit_arg} {remaining}")
            evaluate.append(shlex.join([python, "evaluation.py", *run, "--trace-samples", "1"]))

In [ ]:
for stage, commands in [("train", train), ("evaluate", evaluate)]:
    (jobs / f"{stage}.txt").write_bytes(("\n".join(commands) + "\n").encode())
    if test_mode:
        print(f"{stage}: {len(commands)} test commands")
        continue
    (jobs / "logs").mkdir(exist_ok=True)
    task_count = (len(commands) + commands_per_job - 1) // commands_per_job
    script = f"""#!/bin/bash
#SBATCH --job-name=pLTC_{stage}
#SBATCH --partition=normal
#SBATCH --nodes=1
#SBATCH --ntasks=1
#SBATCH --cpus-per-task=1
#SBATCH --mem=1G
#SBATCH --array=0-{task_count-1}
#SBATCH --output=jobs/logs/%x-%A_%a.out
#SBATCH --error=jobs/logs/%x-%A_%a.err
set -eu
cd "$SLURM_SUBMIT_DIR"
first=$((SLURM_ARRAY_TASK_ID * {commands_per_job} + 1))
last=$((first + {commands_per_job} - 1))
failed=0
while IFS= read -r command; do
    bash -c "$command" || failed=1
done < <(sed -n "${{first}},${{last}}p" jobs/{stage}.txt)
exit "$failed"
"""
    (jobs / f"{stage}.slurm").write_bytes(script.encode())
    print(f"{stage}: {len(commands)} experiments, {task_count} array tasks")